In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import dok_matrix, save_npz
import os
import json

In [ ]:
OUTPUT_PATH = "./DDDB/DrugToDisease_DGIDB_naming.tsv"
#get the claim_name to name correspondence
DGIDB = pd.read_csv("./DGIDB/DrugToGene.tsv", sep="\t")
DDDB = pd.read_csv("./DDDB/DrugToDisease.tsv", sep="\t")

#normalize columns
DGIDB['drug_name'] = DGIDB['drug_name'].str.upper()
DGIDB['drug_claim_name'] = DGIDB['drug_claim_name'].str.upper()

DDDB['ndfrt_preferred_label'] = DDDB['ndfrt_preferred_label'].str.upper()
DDDB_drug_name_array = DDDB['ndfrt_preferred_label'].unique()
print(DDDB_drug_name_array)

DDDB.iloc[38:51]

In [ ]:
#building correspondence
claim2canon = (
    DGIDB.dropna(subset=["drug_claim_name", "drug_name"])
         .drop_duplicates("drug_claim_name")
         .set_index("drug_claim_name")["drug_name"]
         .to_dict()
)

DDDB["ndfrt_preferred_label"] = (
    DDDB["ndfrt_preferred_label"].map(claim2canon).fillna(DDDB["ndfrt_preferred_label"])
)



In [ ]:
#view result
DDDB.iloc[38:51]

In [ ]:
#view missing drug names

# Normalize to lowercase for fair comparison
DGIDB_names = set(DGIDB['drug_name'].str.lower())

# Filter DDDB to keep only the "missing" ones
missing_rows = DDDB[~DDDB['ndfrt_preferred_label'].str.lower().isin(DGIDB_names)]

# Show them
missing_unique = (
    missing_rows['ndfrt_preferred_label']
    .drop_duplicates()
    .sort_values()
)
print(missing_unique.to_string(index=False))


In [ ]:
#create the updated DDDB tsv file
DDDB.to_csv(OUTPUT_PATH, sep="\t", index=False)